# Day 31: LLMOps Intro and FDE Skills 4
**The LLM lifecycle, what breaks in production, escalations and emotional intelligence**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
### How to use this notebook
* Run a cell with **Shift + Enter**.
* Some cells have two versions: **Option A (Google Colab)** is active, **Option B (VS Code / local)** is commented out.
  * In Colab: run as is.
  * In VS Code: comment Option A (select lines, press `Ctrl + /` or `Cmd + /`) and uncomment Option B.
* Cells marked **TODO** are for you to complete.

## 1. Setup

In [ ]:
# ===== Option A: Google Colab (active) =====
# Add your key once: click the key icon (Secrets) in the left sidebar, name it OPENAI_API_KEY, and allow notebook access.
!pip install -q openai scikit-learn numpy pandas matplotlib

# ===== Option B: VS Code / local =====
# pip install openai scikit-learn numpy pandas matplotlib
# Create a .env file or set the variable in your terminal:  OPENAI_API_KEY=sk-...
# No key? Everything still runs in OFFLINE demo mode with a tiny built-in stand-in model.
print("Setup done")

In [ ]:
%%writefile genai_helpers.py
"""Shared helpers for the GenAI days.
llm()   -> calls OpenAI when a key is available, otherwise a small OFFLINE stand-in so every cell still runs.
embed() -> OpenAI embeddings online, a hashing-based stand-in offline (same API, lower quality).
"""
import json, os, re
import numpy as np

def _get_key():
    try:
        from google.colab import userdata          # Colab Secrets
        return userdata.get("OPENAI_API_KEY")
    except Exception:
        return os.getenv("OPENAI_API_KEY")

OPENAI_API_KEY = _get_key()
ONLINE = bool(OPENAI_API_KEY)
MODEL = os.getenv("LLM_MODEL", "gpt-4o-mini")                 # change to any current chat model you have access to
EMBED_MODEL = os.getenv("EMBED_MODEL", "text-embedding-3-small")
if ONLINE:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)

WORD = re.compile(r"[a-z0-9]+")

def _words(t):
    # crude normalisation so "expires" matches "expire"
    return {w[:-1] if w.endswith("s") and len(w) > 4 else w for w in WORD.findall(t.lower()) if len(w) > 3}

def offline_llm(prompt, system=""):
    """A deliberately simple stand-in. It is NOT intelligent; it lets the code paths run without a key."""
    low = prompt.lower()
    if "context:" in low and "question:" in low:
        ctx = re.split(r"(?i)context:", prompt, 1)[1]
        ctx, q = re.split(r"(?i)question:", ctx, 1)
        sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", ctx.strip()) if s.strip()]
        best = max(sents, key=lambda s: len(_words(s) & _words(q)), default="")
        return best if best and _words(best) & _words(q) else "I don't know based on the provided context."
    if "classify" in low and "positive" in low:
        neg = sum(w in low for w in ["slow", "bad", "broken", "late", "angry", "worst", "refund"])
        pos = sum(w in low for w in ["great", "love", "fast", "thanks", "excellent", "happy"])
        return "negative" if neg > pos else "positive" if pos > neg else "neutral"
    if "json" in low:
        text = prompt.strip().splitlines()[-1]
        return json.dumps({"summary": " ".join(text.split()[:12]), "offline": True})
    first = re.split(r"(?<=[.!?])\s+", prompt.strip())[0]
    return f"[offline demo] {first[:160]}"

def llm(prompt, system="You are a helpful assistant.", temperature=0.2, model=None):
    if ONLINE:
        r = client.chat.completions.create(
            model=model or MODEL, temperature=temperature,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": prompt}])
        return r.choices[0].message.content
    return offline_llm(prompt, system)

def embed(texts):
    """Return an (n, d) array of unit-length vectors."""
    if isinstance(texts, str):
        texts = [texts]
    if ONLINE:
        r = client.embeddings.create(model=EMBED_MODEL, input=texts)
        v = np.array([d.embedding for d in r.data])
    else:
        from sklearn.feature_extraction.text import HashingVectorizer
        hv = HashingVectorizer(n_features=1024, ngram_range=(1, 2), alternate_sign=False, norm=None)
        v = hv.transform([t.lower() for t in texts]).toarray()
    n = np.linalg.norm(v, axis=1, keepdims=True)
    return v / np.where(n == 0, 1, n)

print("Mode:", f"ONLINE ({MODEL})" if ONLINE else "OFFLINE demo (add OPENAI_API_KEY to use a real model)")

In [ ]:
import importlib, genai_helpers
importlib.reload(genai_helpers)
from genai_helpers import llm, embed, ONLINE

## 2. Incident simulation: a week of LLM request logs

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
rng = np.random.default_rng(31)
n = 8000
ts = pd.Timestamp("2026-11-30") + pd.to_timedelta(rng.integers(0, 7 * 24 * 60, n), unit="min")
logs = pd.DataFrame({"ts": ts, "model": "main-model-v1", "status": 200,
                     "latency_ms": rng.gamma(4, 300, n).round(), "tokens": rng.normal(900, 150, n).round()})
day, hour = logs.ts.dt.date.astype(str), logs.ts.dt.hour
burst = (day == "2026-12-02") & hour.between(10, 10)                   # rate limit storm
logs.loc[burst & (rng.random(n) < 0.55), "status"] = 429
logs.loc[rng.random(n) < 0.01, "status"] = 503                          # background provider errors
logs.loc[logs.ts >= "2026-12-04", "tokens"] *= 3                        # a prompt change added a huge context
logs.loc[logs.status != 200, "latency_ms"] = rng.integers(20000, 30000, (logs.status != 200).sum())
logs = logs.sort_values("ts").reset_index(drop=True)
print(logs.head())

In [ ]:
by_hour = logs.set_index("ts").resample("1h").agg(errors=("status", lambda s: (s != 200).mean()), requests=("status", "size"))
worst = by_hour.errors.idxmax()
print(f"Worst hour: {worst} with {by_hour.errors.max():.0%} errors")
print("Status codes in that hour:", logs[(logs.ts >= worst) & (logs.ts < worst + pd.Timedelta('1h'))].status.value_counts().to_dict())

daily_tokens = logs.groupby(logs.ts.dt.date)["tokens"].mean().round()
print("\nAverage tokens per request by day:\n", daily_tokens.to_string())

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
by_hour.errors.plot(ax=ax[0], color="#FF8A3D", title="Error rate per hour"); ax[0].set_ylabel("share of requests")
daily_tokens.plot(ax=ax[1], kind="bar", color="#0FA3A3", title="Average tokens per request"); plt.tight_layout(); plt.show()

## 3. A flaky provider and resilience patterns
The simulated provider fails 30% of the time with transient errors. Waits are scaled down so the demo runs fast.

In [ ]:
import random, time

class Transient(Exception):
    pass

class FlakyProvider:
    def __init__(self, fail_rate, name):
        self.fail_rate, self.name, self.calls = fail_rate, name, 0
    def complete(self, prompt):
        self.calls += 1
        if random.random() < self.fail_rate:
            raise Transient(f"{self.name}: 429 or timeout")
        return f"[{self.name}] answer to: {prompt[:30]}"

def with_retry(fn, retries=3, base=0.001):
    for attempt in range(retries + 1):
        try:
            return fn()
        except Transient:
            if attempt == retries:
                raise
            time.sleep(base * (2 ** attempt) + random.random() * base)    # exponential backoff with jitter

random.seed(31)
main = FlakyProvider(0.30, "main")
for retries in [0, 2, 4]:
    ok = 0
    for i in range(500):
        try:
            with_retry(lambda: main.complete("hello"), retries=retries); ok += 1
        except Transient:
            pass
    print(f"retries={retries}: success {ok / 500:.0%}")

In [ ]:
class CircuitBreaker:
    def __init__(self, threshold=3, cooldown=0.05):
        self.threshold, self.cooldown = threshold, cooldown
        self.failures, self.opened_at, self.state = 0, None, "CLOSED"
        self.log = ["CLOSED"]

    def _set(self, state):
        if state != self.state:
            self.state = state
            self.log.append(state)

    def call(self, fn):
        if self.state == "OPEN":
            if time.time() - self.opened_at < self.cooldown:
                raise Transient("circuit open")
            self._set("HALF-OPEN")
        try:
            result = fn()
            self.failures = 0
            self._set("CLOSED")
            return result
        except Transient:
            self.failures += 1
            if self.failures >= self.threshold or self.state == "HALF-OPEN":
                self._set("OPEN")
                self.opened_at = time.time()
            raise

random.seed(7)
outage = FlakyProvider(0.95, "main")          # the main provider is having an outage
backup = FlakyProvider(0.02, "backup")
breaker = CircuitBreaker()
served = {"main": 0, "backup": 0, "error message": 0}
states = []
for i in range(40):
    time.sleep(0.005)                          # requests arrive over time, so the cool-down can expire
    try:
        breaker.call(lambda: outage.complete("Where is my order?")); served["main"] += 1
    except Transient:
        try:
            with_retry(lambda: backup.complete("Where is my order?"), retries=1); served["backup"] += 1
        except Transient:
            served["error message"] += 1
    states.append(breaker.state)
print("Served by:", served)
print("Calls that reached the failing main provider:", outage.calls, "out of 40 requests")
print("Breaker transitions:", " -> ".join(breaker.log[:9]), "...")

### TODO 1: Make the circuit breaker log every state change with a timestamp, and add a `served_from_cache` option for frequent questions.

In [ ]:
# TODO: write your code here

## 4. Escalation message builder

In [ ]:
incident = {
    "severity": "SEV2", "what": "Support assistant failing for 55% of requests",
    "since": "2026-12-02 10:05 IST", "cause": "provider rate limits (HTTP 429)",
    "impact": "about 1,200 customers saw an error message",
    "actions": ["enabled exponential backoff", "routed overflow to the backup model", "error rate now 8%"],
    "ask": "approve the rate-limit increase request with the provider today",
    "owner": "Ravi (platform lead)", "next_update": "11:30 IST",
}
def escalation(i):
    return (f"[{i['severity']}] {i['what']} since {i['since']} due to {i['cause']}.\n"
            f"Impact: {i['impact']}.\n"
            f"Actions so far: {'; '.join(i['actions'])}.\n"
            f"Ask: {i['owner']}, please {i['ask']}.\n"
            f"Next update: {i['next_update']}, even if nothing changes.")
print(escalation(incident))

## 5. Emotional intelligence: rewriting defensive messages

In [ ]:
defensive = ["It is the provider's outage, not our fault.",
             "We followed the spec, so this is not our problem.",
             "I already explained this yesterday."]
EQ_SYSTEM = "Rewrite the message for an upset customer: acknowledge their impact, own the next step, stay factual, one or two sentences."
examples = {   # offline: model answers written by a person, for comparison
    defensive[0]: "I understand this outage hurt your team today. We have switched to a backup model and I will update you at 11:30.",
    defensive[1]: "I hear that the result is not what you need. Let us look at the gap together today and agree the fix.",
    defensive[2]: "Thanks for raising it again; I clearly did not explain it well. Here it is in two lines, and I will send it in writing too.",
}
for d in defensive:
    better = llm(d, system=EQ_SYSTEM) if ONLINE else examples[d]
    print(f"Before: {d}\nAfter:  {better}\n")

---
**Ganbatte** (頑張って): *Do your best*

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)